In [16]:
import requests
from datetime import datetime, timedelta
from fastapi import FastAPI, HTTPException
import joblib
import numpy as np

In [17]:
# 固定开始时间
start_time = datetime(2025, 5, 1)  # 2025年5月1日 00:00:00

# 结束时间为当前时间
end_time = datetime.now()

In [18]:
names = ['DLDZ_DQ200_KYJ01_YI01.PV','DLDZ_DQ200_KYJ02_YI02.PV','DLDZ_DQ200_KYJ03_YI02.PV',
         'DLDZ_DQ200_KYJ04_YI02.PV','DLDZ_DQ200_KYJ05_YI02.PV','DLDZ_DQ200_KYJ06_YI02.PV',
         'DLDZ_AVS_KYJ01_YI01.PV','DLDZ_AVS_KYJ02_YI01.PV','DLDZ_AVS_KYJ03_YI01.PV',
         'DLDZ_AVS_KYJ04_YI01.PV','DLDZ_AVS_KYJ05_YI01.PV',
         'DLDZ_DQ200_LLJ01_FI01.PV','DLDZ_AVS_LLJ01_FI01.PV'
        ]

In [ ]:
# DQ200
names = ['DLDZ_DQ200_KYJ02_YI02.PV', 'DLDZ_DQ200_KYJ03_YI02.PV',
          'DLDZ_DQ200_KYJ04_YI02.PV', 'DLDZ_DQ200_KYJ05_YI02.PV',
        'DLDZ_DQ200_KYJ06_YI02.PV'
        ]

In [ ]:
# AVS
names = ['DLDZ_AVS_KYJ01_YI01.PV','DLDZ_AVS_KYJ02_YI01.PV','DLDZ_AVS_KYJ03_YI01.PV',
         'DLDZ_AVS_KYJ04_YI01.PV','DLDZ_AVS_KYJ05_YI01.PV']

In [47]:
# 瞬时流量
names = ['DLDZ_DQ200_LLJ01_FI01.PV',
'DLDZ_AVS_LLJ01_FI01.PV']

In [19]:
# 构建API请求参数
params = {
    "startTime": start_time.isoformat(timespec='milliseconds'),
    "endTime": end_time.isoformat(timespec='milliseconds'),
    "interval": 3600000,  # 1小时=3600000ms, 1分钟=60000ms
    "valueonly": 0,
    "decimal": 2,
    "names": ','.join(names)
}

In [20]:
# 发送GET请求
try:
    response = requests.get(
        "http://8.130.25.118:8000/api/hisdata",
        params=params,
        timeout=10  # 设置超时时间
    )
    response.raise_for_status()  # 检查HTTP状态码
except Exception as e:
    raise Exception(f"API请求失败: {str(e)}")
# 解析响应数据
data = response.json()

In [21]:
for i in range(13):
    print(data['items'][i]['name'])
    print(len(data['items'][i]['vals']))

DLDZ_DQ200_KYJ01_YI01.PV
704
DLDZ_DQ200_KYJ02_YI02.PV
704
DLDZ_DQ200_KYJ03_YI02.PV
704
DLDZ_DQ200_KYJ04_YI02.PV
704
DLDZ_DQ200_KYJ05_YI02.PV
704
DLDZ_DQ200_KYJ06_YI02.PV
704
DLDZ_AVS_KYJ01_YI01.PV
704
DLDZ_AVS_KYJ02_YI01.PV
704
DLDZ_AVS_KYJ03_YI01.PV
704
DLDZ_AVS_KYJ04_YI01.PV
704
DLDZ_AVS_KYJ05_YI01.PV
704
DLDZ_DQ200_LLJ01_FI01.PV
704
DLDZ_AVS_LLJ01_FI01.PV
704


In [21]:
print(data['items'][1]['name'])
print(len(data['items'][1]['vals']))

DLDZ_DQ200_KYJ02_YI02.PV
0


In [7]:
data

{'code': 0,
 'items': [{'name': 'DLDZ_DQ200_KYJ01_YI01.PV',
   'vals': [{'time': '2025-05-01T00:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T01:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T02:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T03:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T04:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T05:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T06:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T07:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T08:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T09:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T10:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T11:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T12:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T13:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T14:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T15:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T16:00:00.000', 'val': 0.0},
    {'time': '2025-05-01T17:00:

In [9]:
first_item = data["items"][1]
first_item

{'name': 'DLDZ_DQ200_KYJ02_YI02.PV', 'vals': []}

In [22]:
first_item = data["items"][1]
print("测点名称:", first_item["name"])  # 应输出类似 'DLDZ_XXX.PV'
print("测点的第一个数据点:", first_item["vals"][0])  # 应输出 {'time': ..., 'val': ...}

测点名称: DLDZ_DQ200_KYJ02_YI02.PV
测点的第一个数据点: {'time': '2025-05-01T00:00:00.000', 'val': -9999.0}


In [23]:
import pandas as pd


# 创建主DataFrame（存储最终结果）
master_df = pd.DataFrame()

for item in data["items"]:
    # 确保每个测点有 'name' 和 'vals'
    if "name" not in item or "vals" not in item:
        print(f"无效的测点数据: {item}")
        continue
    
    site_name = item["name"]
    vals = item["vals"]
    
    # 检查 vals 是否非空且包含 'time' 和 'val'
    if not vals or any("time" not in entry or "val" not in entry for entry in vals):
        print(f"测点 {site_name} 的数据格式错误或缺失字段")
        continue
    
    # 转换为DataFrame
    df = pd.DataFrame(vals)
    
    # 验证列是否存在
    if "time" not in df.columns or "val" not in df.columns:
        print(f"测点 {site_name} 的 DataFrame 缺少 'time' 或 'val' 列")
        continue
    
    # 转换时间格式并设置索引
    try:
        df["time"] = pd.to_datetime(df["time"]).dt.strftime("%Y-%m-%d %H:%M:%S")
        df_single = df.set_index("time")[["val"]].rename(columns={"val": site_name})
    except Exception as e:
        print(f"处理测点 {site_name} 时出错: {str(e)}")
        continue
    
    # 合并到主表
    if master_df.empty:
        master_df = df_single
    else:
        master_df = master_df.join(df_single, how="outer")

# 重置索引并将时间移到第一列
master_df = master_df.reset_index().rename(columns={"index": "time"})

# 保存为Excel文件
master_df.to_excel("点位信息.xlsx", index=False)